# Computational Notebook 04: Smart Contract Development

## Overview

Smart contracts are self-executing programs stored on a blockchain that automatically enforce agreements when predefined conditions are met. This notebook explores smart contract development concepts by building Python simulators for Solidity patterns, token standards (ERC-20, ERC-721), security vulnerabilities, and gas optimization. All implementations run in pure Python without requiring any blockchain infrastructure.

## Prerequisites
- Notebook 01: Cryptographic Primitives (hash functions, digital signatures)
- Notebook 03: Ethereum & EVM Analysis (EVM architecture, gas model)
- Basic Python programming (classes, decorators, dataclasses)

## Learning Objectives
1. Understand Solidity data types and how they map to Python equivalents
2. Implement common contract design patterns (Ownable, Pausable, ReentrancyGuard)
3. Build a complete ERC-20 token with transfer, approve, and transferFrom
4. Build an ERC-721 NFT contract with minting and ownership tracking
5. Identify and demonstrate critical security vulnerabilities (reentrancy, overflow)
6. Analyze gas costs and apply optimization techniques

**Estimated Time:** 4-6 hours

**Related Content:** [Section 03: Ethereum & Smart Contracts](../sections/03-ethereum-smart-contracts.md)

In [ ]:
# Setup and imports
import hashlib
import json
import time
from typing import Dict, List, Optional, Tuple, Any, Callable
from dataclasses import dataclass, field
from collections import defaultdict
from enum import Enum
import copy

print("All imports successful!")
print("This notebook uses pure Python to simulate smart contract concepts.")
print("No Solidity compiler or blockchain connection required.")

---
## 1. Solidity Fundamentals: Data Types and Storage

Solidity, Ethereum's primary smart contract language, has specific data types optimized for the EVM.
Let's simulate these types and understand their constraints.

### Key Solidity Types
| Solidity Type | Size | Python Equivalent | Notes |
|:---|:---|:---|:---|
| `uint256` | 32 bytes | `int` (bounded) | Unsigned, 0 to 2^256-1 |
| `int256` | 32 bytes | `int` (bounded) | Signed, -2^255 to 2^255-1 |
| `address` | 20 bytes | `str` (hex) | 0x-prefixed, 40 hex chars |
| `bool` | 1 byte | `bool` | True/False |
| `bytes32` | 32 bytes | `bytes` | Fixed-size byte array |
| `mapping` | - | `dict` | Key-value storage |
| `string` | variable | `str` | Dynamic-size UTF-8 |

In [ ]:
class SolidityTypes:
    """Simulate Solidity type constraints in Python."""
    
    UINT256_MAX = 2**256 - 1
    INT256_MAX = 2**255 - 1
    INT256_MIN = -(2**255)
    
    @staticmethod
    def uint256(value: int) -> int:
        """Validate and return a uint256 value."""
        if not isinstance(value, int) or value < 0:
            raise ValueError(f"uint256 must be non-negative integer, got {value}")
        if value > SolidityTypes.UINT256_MAX:
            raise OverflowError(f"uint256 overflow: {value} > 2^256-1")
        return value
    
    @staticmethod
    def int256(value: int) -> int:
        """Validate and return an int256 value."""
        if value > SolidityTypes.INT256_MAX or value < SolidityTypes.INT256_MIN:
            raise OverflowError(f"int256 overflow: {value} outside range")
        return value
    
    @staticmethod
    def address(value: str) -> str:
        """Validate and return an Ethereum address."""
        if not value.startswith("0x") or len(value) != 42:
            raise ValueError(f"Invalid address format: {value}")
        try:
            int(value, 16)
        except ValueError:
            raise ValueError(f"Address contains non-hex characters: {value}")
        return value.lower()
    
    @staticmethod
    def bytes32(value: bytes) -> bytes:
        """Validate and return a bytes32 value."""
        if len(value) > 32:
            raise ValueError(f"bytes32 overflow: {len(value)} bytes")
        return value.ljust(32, b'\x00')  # Right-pad with zeros


# Demonstrate type constraints
print("=" * 60)
print("SOLIDITY TYPE SIMULATION")
print("=" * 60)

# uint256
print(f"\nuint256(100) = {SolidityTypes.uint256(100)}")
print(f"uint256 MAX  = 2^256 - 1 = {SolidityTypes.UINT256_MAX}")
print(f"  (that's {len(str(SolidityTypes.UINT256_MAX))} digits!)")

# Test overflow
try:
    SolidityTypes.uint256(-1)
except ValueError as e:
    print(f"\nuint256(-1) -> ValueError: {e}")

try:
    SolidityTypes.uint256(2**256)
except OverflowError as e:
    print(f"uint256(2^256) -> OverflowError: {e}")

# address
addr = SolidityTypes.address("0x742d35Cc6634C0532925a3b844Bc9e7595f2bD58")
print(f"\nValid address: {addr}")

try:
    SolidityTypes.address("0xinvalid")
except ValueError as e:
    print(f"Invalid address -> {e}")

### Solidity Storage Model

In the EVM, contract storage is a key-value store mapping 256-bit keys to 256-bit values.
Each storage slot costs gas to read (SLOAD = 2,100 gas) and write (SSTORE = 20,000 gas for new, 5,000 for update).

Let's simulate this storage model.

In [ ]:
@dataclass
class StorageSlot:
    """Represents a single EVM storage slot."""
    key: int
    value: int = 0
    dirty: bool = False  # Modified since last commit


class ContractStorage:
    """Simulate EVM contract storage with gas accounting."""
    
    GAS_SLOAD = 2_100     # Cold storage read
    GAS_SSTORE_NEW = 20_000   # Write to empty slot
    GAS_SSTORE_UPDATE = 5_000  # Update existing slot
    GAS_SSTORE_CLEAR = 0       # Refund for clearing (simplified)
    REFUND_CLEAR = 4_800       # Gas refund for clearing storage
    
    def __init__(self) -> None:
        """Initialize empty contract storage."""
        self.slots: Dict[int, StorageSlot] = {}
        self.gas_used = 0
        self.gas_log: List[str] = []
    
    def sload(self, slot: int) -> int:
        """Read from storage slot (costs gas)."""
        self.gas_used += self.GAS_SLOAD
        self.gas_log.append(f"SLOAD slot {slot}: {self.GAS_SLOAD} gas")
        if slot in self.slots:
            return self.slots[slot].value
        return 0
    
    def sstore(self, slot: int, value: int) -> None:
        """Write to storage slot (costs gas)."""
        if slot in self.slots:
            if value == 0 and self.slots[slot].value != 0:
                gas = self.GAS_SSTORE_CLEAR
                self.gas_log.append(f"SSTORE slot {slot} = 0 (clear): {gas} gas + {self.REFUND_CLEAR} refund")
            else:
                gas = self.GAS_SSTORE_UPDATE
                self.gas_log.append(f"SSTORE slot {slot} = {value} (update): {gas} gas")
        else:
            gas = self.GAS_SSTORE_NEW
            self.gas_log.append(f"SSTORE slot {slot} = {value} (new): {gas} gas")
        
        self.gas_used += gas
        self.slots[slot] = StorageSlot(key=slot, value=value, dirty=True)


# Demonstrate storage gas costs
storage = ContractStorage()

print("=" * 60)
print("EVM STORAGE GAS SIMULATION")
print("=" * 60)

# Write to new slot (expensive)
storage.sstore(0, 42)

# Read from slot
val = storage.sload(0)

# Update existing slot (cheaper)
storage.sstore(0, 100)

# Write to another new slot
storage.sstore(1, 200)

# Clear a slot (refund)
storage.sstore(0, 0)

print("\nOperation Log:")
for entry in storage.gas_log:
    print(f"  {entry}")
print(f"\nTotal gas used: {storage.gas_used:,}")

### SimpleStorage Contract Simulator

Let's build our first complete contract simulator - equivalent to the classic Solidity SimpleStorage:

```solidity
contract SimpleStorage {
    uint256 storedData;
    
    function set(uint256 x) public {
        storedData = x;
    }
    
    function get() public view returns (uint256) {
        return storedData;
    }
}
```

In [ ]:
class ContractBase:
    """Base class for all simulated smart contracts."""
    
    def __init__(self, deployer: str) -> None:
        """Deploy the contract from a given address."""
        self.address = "0x" + hashlib.sha256(f"{deployer}{time.time()}".encode()).hexdigest()[:40]
        self.deployer = deployer
        self.storage = ContractStorage()
        self.events: List[Dict[str, Any]] = []
        self.msg_sender = deployer  # Current caller
    
    def _emit_event(self, name: str, **kwargs: Any) -> None:
        """Emit a contract event."""
        event = {"event": name, "sender": self.msg_sender, **kwargs}
        self.events.append(event)
    
    def _require(self, condition: bool, message: str = "Requirement failed") -> None:
        """Solidity-style require statement."""
        if not condition:
            raise Exception(f"REVERT: {message}")
    
    def call_as(self, sender: str) -> 'ContractBase':
        """Set msg.sender for the next call."""
        self.msg_sender = sender
        return self


class SimpleStorage(ContractBase):
    """Python simulation of a SimpleStorage Solidity contract."""
    
    def __init__(self, deployer: str) -> None:
        """Deploy SimpleStorage contract."""
        super().__init__(deployer)
        self._stored_data = 0
        self._emit_event("ContractDeployed", contract=self.address)
    
    def set(self, value: int) -> None:
        """Store a uint256 value."""
        SolidityTypes.uint256(value)  # Type check
        old_value = self._stored_data
        self._stored_data = value
        self._emit_event("ValueChanged", old_value=old_value, new_value=value)
    
    def get(self) -> int:
        """Retrieve the stored value."""
        return self._stored_data


# Deploy and interact with SimpleStorage
deployer = "0x" + "a1" * 20
contract = SimpleStorage(deployer)

print("=" * 60)
print("SIMPLE STORAGE CONTRACT")
print("=" * 60)
print(f"Deployed at: {contract.address}")
print(f"Deployer:    {deployer}")
print(f"\nInitial value: {contract.get()}")

contract.set(42)
print(f"After set(42): {contract.get()}")

contract.set(999)
print(f"After set(999): {contract.get()}")

print(f"\nEvents emitted:")
for event in contract.events:
    print(f"  {event}")

---
## 2. Common Contract Design Patterns

Smart contracts use well-established design patterns for security and functionality.
These patterns are battle-tested through frameworks like OpenZeppelin.

### 2.1 Ownable Pattern
Restricts certain functions to the contract owner. This is the most fundamental access control pattern.

In [ ]:
class Ownable(ContractBase):
    """Ownable contract pattern - restricts access to owner."""
    
    def __init__(self, deployer: str) -> None:
        """Deploy with deployer as initial owner."""
        super().__init__(deployer)
        self._owner = deployer
        self._emit_event("OwnershipTransferred", previous_owner="0x" + "00" * 20, new_owner=deployer)
    
    def _only_owner(self) -> None:
        """Modifier: restrict to owner only."""
        self._require(self.msg_sender == self._owner, "Ownable: caller is not the owner")
    
    def owner(self) -> str:
        """Return the current owner."""
        return self._owner
    
    def transfer_ownership(self, new_owner: str) -> None:
        """Transfer ownership to a new address."""
        self._only_owner()
        self._require(new_owner != "0x" + "00" * 20, "Ownable: new owner is zero address")
        old_owner = self._owner
        self._owner = new_owner
        self._emit_event("OwnershipTransferred", previous_owner=old_owner, new_owner=new_owner)
    
    def renounce_ownership(self) -> None:
        """Renounce ownership (irreversible)."""
        self._only_owner()
        old_owner = self._owner
        self._owner = "0x" + "00" * 20
        self._emit_event("OwnershipTransferred", previous_owner=old_owner, new_owner=self._owner)


# Demonstrate Ownable
owner_addr = "0x" + "a1" * 20
user_addr = "0x" + "b2" * 20

contract = Ownable(owner_addr)

print("=" * 60)
print("OWNABLE PATTERN")
print("=" * 60)
print(f"Owner: {contract.owner()[:12]}...")

# Owner can transfer
contract.call_as(owner_addr).transfer_ownership(user_addr)
print(f"After transfer -> Owner: {contract.owner()[:12]}...")

# Non-owner cannot transfer
try:
    contract.call_as(owner_addr).transfer_ownership(owner_addr)
except Exception as e:
    print(f"Non-owner attempt: {e}")

# New owner can renounce
contract.call_as(user_addr).renounce_ownership()
print(f"After renounce  -> Owner: {contract.owner()[:12]}... (zero address)")

### 2.2 Pausable Pattern

Allows the owner to pause and unpause contract operations - a critical emergency stop mechanism.

In [ ]:
class Pausable(Ownable):
    """Pausable contract - adds emergency stop functionality."""
    
    def __init__(self, deployer: str) -> None:
        """Deploy in unpaused state."""
        super().__init__(deployer)
        self._paused = False
    
    def _when_not_paused(self) -> None:
        """Modifier: require contract is not paused."""
        self._require(not self._paused, "Pausable: paused")
    
    def _when_paused(self) -> None:
        """Modifier: require contract is paused."""
        self._require(self._paused, "Pausable: not paused")
    
    def paused(self) -> bool:
        """Return whether the contract is paused."""
        return self._paused
    
    def pause(self) -> None:
        """Pause the contract (owner only)."""
        self._only_owner()
        self._when_not_paused()
        self._paused = True
        self._emit_event("Paused", account=self.msg_sender)
    
    def unpause(self) -> None:
        """Unpause the contract (owner only)."""
        self._only_owner()
        self._when_paused()
        self._paused = False
        self._emit_event("Unpaused", account=self.msg_sender)


# Demonstrate Pausable
owner_addr = "0x" + "a1" * 20
contract = Pausable(owner_addr)

print("=" * 60)
print("PAUSABLE PATTERN")
print("=" * 60)
print(f"Paused: {contract.paused()}")

contract.call_as(owner_addr).pause()
print(f"After pause(): {contract.paused()}")

contract.call_as(owner_addr).unpause()
print(f"After unpause(): {contract.paused()}")

# Non-owner cannot pause
user_addr = "0x" + "b2" * 20
try:
    contract.call_as(user_addr).pause()
except Exception as e:
    print(f"Non-owner pause: {e}")

### 2.3 ReentrancyGuard Pattern

Prevents reentrant calls to a function. This is critical for functions that make external calls, as reentrancy is one of the most dangerous smart contract vulnerabilities (as seen in the 2016 DAO hack).

In [ ]:
class ReentrancyGuard:
    """Reentrancy guard mixin - prevents recursive calls."""
    
    _NOT_ENTERED = 1
    _ENTERED = 2
    
    def __init__(self) -> None:
        """Initialize guard state."""
        self._reentrancy_status = self._NOT_ENTERED
    
    def _nonreentrant_enter(self) -> None:
        """Enter nonreentrant block."""
        if self._reentrancy_status == self._ENTERED:
            raise Exception("REVERT: ReentrancyGuard: reentrant call")
        self._reentrancy_status = self._ENTERED
    
    def _nonreentrant_exit(self) -> None:
        """Exit nonreentrant block."""
        self._reentrancy_status = self._NOT_ENTERED


# Demonstrate the guard
guard = ReentrancyGuard()

print("=" * 60)
print("REENTRANCY GUARD PATTERN")
print("=" * 60)

# Normal usage
guard._nonreentrant_enter()
print("Entered protected function")
guard._nonreentrant_exit()
print("Exited protected function")

# Reentrant call attempt
guard._nonreentrant_enter()
try:
    guard._nonreentrant_enter()  # Simulate reentrant call
except Exception as e:
    print(f"Reentrant call blocked: {e}")
guard._nonreentrant_exit()

---
## 3. ERC-20 Token Standard

ERC-20 is the most widely adopted token standard on Ethereum. It defines a common interface for fungible tokens, enabling interoperability across wallets, exchanges, and DeFi protocols.

### ERC-20 Interface (Solidity)
```solidity
interface IERC20 {
    function totalSupply() external view returns (uint256);
    function balanceOf(address account) external view returns (uint256);
    function transfer(address to, uint256 amount) external returns (bool);
    function allowance(address owner, address spender) external view returns (uint256);
    function approve(address spender, uint256 amount) external returns (bool);
    function transferFrom(address from, address to, uint256 amount) external returns (bool);
}
```

Let's implement a complete ERC-20 token in Python.

In [ ]:
class ERC20(ContractBase):
    """Complete ERC-20 token implementation in Python."""
    
    ZERO_ADDRESS = "0x" + "00" * 20
    
    def __init__(self, deployer: str, name: str, symbol: str,
                 decimals: int = 18, initial_supply: int = 0) -> None:
        """Deploy ERC-20 token with initial supply minted to deployer."""
        super().__init__(deployer)
        self._name = name
        self._symbol = symbol
        self._decimals = decimals
        self._total_supply = 0
        self._balances: Dict[str, int] = defaultdict(int)
        self._allowances: Dict[str, Dict[str, int]] = defaultdict(lambda: defaultdict(int))
        
        if initial_supply > 0:
            self._mint(deployer, initial_supply)
    
    # --- View Functions ---
    
    def name(self) -> str:
        """Return the token name."""
        return self._name
    
    def symbol(self) -> str:
        """Return the token symbol."""
        return self._symbol
    
    def decimals(self) -> int:
        """Return the number of decimals."""
        return self._decimals
    
    def total_supply(self) -> int:
        """Return total token supply."""
        return self._total_supply
    
    def balance_of(self, account: str) -> int:
        """Return the token balance of an account."""
        return self._balances[account]
    
    def allowance(self, owner: str, spender: str) -> int:
        """Return the remaining allowance for a spender."""
        return self._allowances[owner][spender]
    
    # --- State-Changing Functions ---
    
    def transfer(self, to: str, amount: int) -> bool:
        """Transfer tokens from msg.sender to recipient."""
        self._transfer(self.msg_sender, to, amount)
        return True
    
    def approve(self, spender: str, amount: int) -> bool:
        """Approve spender to spend tokens on behalf of msg.sender."""
        self._approve(self.msg_sender, spender, amount)
        return True
    
    def transfer_from(self, from_addr: str, to: str, amount: int) -> bool:
        """Transfer tokens from one address to another using allowance."""
        current_allowance = self._allowances[from_addr][self.msg_sender]
        self._require(current_allowance >= amount, "ERC20: insufficient allowance")
        self._approve(from_addr, self.msg_sender, current_allowance - amount)
        self._transfer(from_addr, to, amount)
        return True
    
    # --- Internal Functions ---
    
    def _transfer(self, from_addr: str, to: str, amount: int) -> None:
        """Internal transfer logic."""
        self._require(from_addr != self.ZERO_ADDRESS, "ERC20: transfer from zero address")
        self._require(to != self.ZERO_ADDRESS, "ERC20: transfer to zero address")
        self._require(self._balances[from_addr] >= amount, "ERC20: transfer amount exceeds balance")
        
        self._balances[from_addr] -= amount
        self._balances[to] += amount
        self._emit_event("Transfer", from_addr=from_addr[:12], to=to[:12], amount=amount)
    
    def _approve(self, owner: str, spender: str, amount: int) -> None:
        """Internal approve logic."""
        self._require(owner != self.ZERO_ADDRESS, "ERC20: approve from zero address")
        self._require(spender != self.ZERO_ADDRESS, "ERC20: approve to zero address")
        self._allowances[owner][spender] = amount
        self._emit_event("Approval", owner=owner[:12], spender=spender[:12], amount=amount)
    
    def _mint(self, account: str, amount: int) -> None:
        """Create new tokens and assign to account."""
        self._require(account != self.ZERO_ADDRESS, "ERC20: mint to zero address")
        self._total_supply += amount
        self._balances[account] += amount
        self._emit_event("Transfer", from_addr="0x00000000", to=account[:12], amount=amount)
    
    def _burn(self, account: str, amount: int) -> None:
        """Destroy tokens from account."""
        self._require(self._balances[account] >= amount, "ERC20: burn amount exceeds balance")
        self._balances[account] -= amount
        self._total_supply -= amount
        self._emit_event("Transfer", from_addr=account[:12], to="0x00000000", amount=amount)
    
    def _format_amount(self, amount: int) -> str:
        """Format a raw token amount with decimals."""
        whole = amount // (10 ** self._decimals)
        frac = amount % (10 ** self._decimals)
        return f"{whole}.{str(frac).zfill(self._decimals)[:4]}" if frac else f"{whole}.0"

print("ERC-20 contract class defined successfully.")

In [ ]:
# Deploy and test the ERC-20 token
alice = "0x" + "a1" * 20
bob = "0x" + "b2" * 20
charlie = "0x" + "c3" * 20

# Deploy with 1,000,000 tokens (18 decimals)
DECIMALS = 18
INITIAL_SUPPLY = 1_000_000 * 10**DECIMALS

token = ERC20(alice, "CryptoEdu Token", "CEDU", DECIMALS, INITIAL_SUPPLY)

print("=" * 60)
print("ERC-20 TOKEN DEPLOYMENT")
print("=" * 60)
print(f"Name:         {token.name()}")
print(f"Symbol:       {token.symbol()}")
print(f"Decimals:     {token.decimals()}")
print(f"Total Supply: {token._format_amount(token.total_supply())} {token.symbol()}")
print(f"\nAlice balance: {token._format_amount(token.balance_of(alice))} {token.symbol()}")
print(f"Bob balance:   {token._format_amount(token.balance_of(bob))} {token.symbol()}")

In [ ]:
# Test transfers and approvals
print("=" * 60)
print("ERC-20 TRANSFER OPERATIONS")
print("=" * 60)

# Alice transfers 10,000 tokens to Bob
transfer_amount = 10_000 * 10**DECIMALS
token.call_as(alice).transfer(bob, transfer_amount)
print(f"Alice -> Bob: {token._format_amount(transfer_amount)} {token.symbol()}")
print(f"  Alice: {token._format_amount(token.balance_of(alice))}")
print(f"  Bob:   {token._format_amount(token.balance_of(bob))}")

# Alice approves Charlie to spend 5,000 tokens
approve_amount = 5_000 * 10**DECIMALS
token.call_as(alice).approve(charlie, approve_amount)
print(f"\nAlice approves Charlie for: {token._format_amount(approve_amount)}")
print(f"  Allowance: {token._format_amount(token.allowance(alice, charlie))}")

# Charlie transfers 3,000 from Alice to Bob (using allowance)
spend_amount = 3_000 * 10**DECIMALS
token.call_as(charlie).transfer_from(alice, bob, spend_amount)
print(f"\nCharlie spends {token._format_amount(spend_amount)} from Alice -> Bob")
print(f"  Alice: {token._format_amount(token.balance_of(alice))}")
print(f"  Bob:   {token._format_amount(token.balance_of(bob))}")
print(f"  Remaining allowance: {token._format_amount(token.allowance(alice, charlie))}")

# Test insufficient balance
print("\n--- Error Cases ---")
try:
    token.call_as(bob).transfer(alice, INITIAL_SUPPLY)  # More than Bob has
except Exception as e:
    print(f"Overspend: {e}")

# Test insufficient allowance
try:
    token.call_as(charlie).transfer_from(alice, bob, approve_amount)  # More than remaining allowance
except Exception as e:
    print(f"Over-allowance: {e}")

---
## 4. ERC-721 NFT Standard

ERC-721 defines the standard for Non-Fungible Tokens (NFTs) - unique, indivisible tokens. Each token has a distinct `tokenId` and a single owner.

### Key Differences from ERC-20
| Feature | ERC-20 | ERC-721 |
|:---|:---|:---|
| Fungibility | Fungible (interchangeable) | Non-fungible (unique) |
| Divisibility | Divisible (via decimals) | Indivisible (whole tokens) |
| Balance | Amount per address | Set of token IDs per address |
| Transfer | Amount-based | Token ID-based |

In [ ]:
class ERC721(ContractBase):
    """ERC-721 Non-Fungible Token implementation."""
    
    ZERO_ADDRESS = "0x" + "00" * 20
    
    def __init__(self, deployer: str, name: str, symbol: str) -> None:
        """Deploy ERC-721 NFT contract."""
        super().__init__(deployer)
        self._name = name
        self._symbol = symbol
        self._owners: Dict[int, str] = {}          # tokenId -> owner
        self._balances: Dict[str, int] = defaultdict(int)   # owner -> count
        self._token_approvals: Dict[int, str] = {}  # tokenId -> approved
        self._operator_approvals: Dict[str, Dict[str, bool]] = defaultdict(lambda: defaultdict(bool))
        self._token_uris: Dict[int, str] = {}       # tokenId -> URI
        self._next_token_id = 1
    
    def name(self) -> str:
        """Return collection name."""
        return self._name
    
    def symbol(self) -> str:
        """Return collection symbol."""
        return self._symbol
    
    def balance_of(self, owner: str) -> int:
        """Return number of NFTs owned by an address."""
        return self._balances[owner]
    
    def owner_of(self, token_id: int) -> str:
        """Return the owner of a specific NFT."""
        self._require(token_id in self._owners, "ERC721: query for nonexistent token")
        return self._owners[token_id]
    
    def token_uri(self, token_id: int) -> str:
        """Return the metadata URI for a token."""
        self._require(token_id in self._owners, "ERC721: URI query for nonexistent token")
        return self._token_uris.get(token_id, "")
    
    def approve(self, to: str, token_id: int) -> None:
        """Approve an address to transfer a specific NFT."""
        owner = self.owner_of(token_id)
        self._require(self.msg_sender == owner or 
                      self._operator_approvals[owner][self.msg_sender],
                      "ERC721: caller is not owner nor approved")
        self._token_approvals[token_id] = to
        self._emit_event("Approval", owner=owner[:12], approved=to[:12], token_id=token_id)
    
    def set_approval_for_all(self, operator: str, approved: bool) -> None:
        """Approve or revoke an operator for all tokens."""
        self._require(operator != self.msg_sender, "ERC721: approve to caller")
        self._operator_approvals[self.msg_sender][operator] = approved
        self._emit_event("ApprovalForAll", owner=self.msg_sender[:12], 
                        operator=operator[:12], approved=approved)
    
    def transfer_from(self, from_addr: str, to: str, token_id: int) -> None:
        """Transfer an NFT from one address to another."""
        owner = self.owner_of(token_id)
        self._require(owner == from_addr, "ERC721: transfer from incorrect owner")
        
        is_approved = (self.msg_sender == owner or
                       self._token_approvals.get(token_id) == self.msg_sender or
                       self._operator_approvals[owner][self.msg_sender])
        self._require(is_approved, "ERC721: caller is not owner nor approved")
        self._require(to != self.ZERO_ADDRESS, "ERC721: transfer to zero address")
        
        # Clear approval
        if token_id in self._token_approvals:
            del self._token_approvals[token_id]
        
        self._balances[from_addr] -= 1
        self._balances[to] += 1
        self._owners[token_id] = to
        self._emit_event("Transfer", from_addr=from_addr[:12], to=to[:12], token_id=token_id)
    
    def mint(self, to: str, uri: str = "") -> int:
        """Mint a new NFT to an address (owner only for simplicity)."""
        self._require(to != self.ZERO_ADDRESS, "ERC721: mint to zero address")
        token_id = self._next_token_id
        self._next_token_id += 1
        
        self._balances[to] += 1
        self._owners[token_id] = to
        if uri:
            self._token_uris[token_id] = uri
        
        self._emit_event("Transfer", from_addr="0x00000000", to=to[:12], token_id=token_id)
        return token_id

print("ERC-721 contract class defined successfully.")

In [ ]:
# Deploy and test the ERC-721 NFT
alice = "0x" + "a1" * 20
bob = "0x" + "b2" * 20

nft = ERC721(alice, "CryptoEdu NFT Collection", "CNFT")

print("=" * 60)
print("ERC-721 NFT CONTRACT")
print("=" * 60)
print(f"Collection: {nft.name()} ({nft.symbol()})")

# Mint NFTs
nft.call_as(alice)
token1 = nft.mint(alice, "ipfs://QmHash1/metadata.json")
token2 = nft.mint(alice, "ipfs://QmHash2/metadata.json")
token3 = nft.mint(bob, "ipfs://QmHash3/metadata.json")

print(f"\nMinted tokens:")
print(f"  Token {token1}: owner={nft.owner_of(token1)[:12]}... (Alice)")
print(f"  Token {token2}: owner={nft.owner_of(token2)[:12]}... (Alice)")
print(f"  Token {token3}: owner={nft.owner_of(token3)[:12]}... (Bob)")
print(f"\nAlice owns {nft.balance_of(alice)} NFTs")
print(f"Bob owns {nft.balance_of(bob)} NFTs")

# Transfer
print("\n--- Transfer Token 1 from Alice to Bob ---")
nft.call_as(alice).transfer_from(alice, bob, token1)
print(f"Token {token1} owner: {nft.owner_of(token1)[:12]}... (Bob)")
print(f"Alice: {nft.balance_of(alice)} NFTs, Bob: {nft.balance_of(bob)} NFTs")

# Approval flow
print("\n--- Approve and Transfer Token 2 ---")
nft.call_as(alice).approve(bob, token2)
nft.call_as(bob).transfer_from(alice, bob, token2)
print(f"Bob transferred Alice's Token {token2} using approval")
print(f"Alice: {nft.balance_of(alice)} NFTs, Bob: {nft.balance_of(bob)} NFTs")

---
## 5. Security Vulnerabilities

Smart contract security is critical because deployed contracts are immutable and handle real value. Let's examine the most dangerous vulnerabilities with Python simulations.

### 5.1 Reentrancy Attack

The most infamous smart contract vulnerability, responsible for the 2016 DAO hack ($60M lost). It occurs when a contract makes an external call before updating its state, allowing the called contract to re-enter and drain funds.

In [ ]:
class VulnerableBank:
    """A bank contract VULNERABLE to reentrancy attack."""
    
    def __init__(self) -> None:
        """Initialize the vulnerable bank."""
        self.balances: Dict[str, int] = defaultdict(int)
        self.total_eth = 0
        self.call_log: List[str] = []
    
    def deposit(self, sender: str, amount: int) -> None:
        """Deposit ETH into the bank."""
        self.balances[sender] += amount
        self.total_eth += amount
        self.call_log.append(f"deposit({sender[:8]}..., {amount})")
    
    def withdraw(self, sender: str, external_call: Optional[Callable] = None) -> None:
        """Withdraw all ETH - VULNERABLE: sends before updating state."""
        balance = self.balances[sender]
        if balance <= 0:
            return
        
        self.call_log.append(f"withdraw({sender[:8]}...): sending {balance} ETH")
        
        # BUG: External call BEFORE state update!
        if external_call:
            external_call(balance)  # Attacker's callback
        
        # State update happens AFTER external call
        self.balances[sender] -= balance
        self.total_eth -= balance


class AttackerContract:
    """Malicious contract that exploits reentrancy."""
    
    def __init__(self, bank: VulnerableBank, address: str) -> None:
        """Set up the attacker targeting a vulnerable bank."""
        self.bank = bank
        self.address = address
        self.stolen = 0
        self.attack_count = 0
        self.max_attacks = 3  # Limit for demonstration
    
    def attack(self) -> None:
        """Initiate the reentrancy attack."""
        self.bank.withdraw(self.address, self._receive)
    
    def _receive(self, amount: int) -> None:
        """Callback when receiving ETH - re-enters the bank."""
        self.stolen += amount
        self.attack_count += 1
        self.bank.call_log.append(f"  [REENTRANT CALL #{self.attack_count}] received {amount}")
        
        # Re-enter the withdraw function!
        if self.attack_count < self.max_attacks and self.bank.balances[self.address] > 0:
            self.bank.withdraw(self.address, self._receive)


# Demonstrate the attack
print("=" * 60)
print("REENTRANCY ATTACK DEMONSTRATION")
print("=" * 60)

bank = VulnerableBank()

# Honest users deposit
bank.deposit("0xhonest1", 10)
bank.deposit("0xhonest2", 10)
bank.deposit("0xhonest3", 10)

# Attacker deposits 1 ETH
attacker_addr = "0xattacker"
bank.deposit(attacker_addr, 1)
print(f"Bank total: {bank.total_eth} ETH")
print(f"Attacker deposited: {bank.balances[attacker_addr]} ETH")

# Launch attack
attacker = AttackerContract(bank, attacker_addr)
attacker.attack()

print(f"\n--- After Attack ---")
print(f"Attacker stolen: {attacker.stolen} ETH (from 1 ETH deposit!)")
print(f"Bank remaining: {bank.total_eth} ETH")

print(f"\nCall trace:")
for log in bank.call_log:
    print(f"  {log}")

In [ ]:
class SecureBank:
    """Bank contract PROTECTED against reentrancy using checks-effects-interactions."""
    
    def __init__(self) -> None:
        """Initialize the secure bank."""
        self.balances: Dict[str, int] = defaultdict(int)
        self.total_eth = 0
        self._entered = False  # Reentrancy guard
        self.call_log: List[str] = []
    
    def deposit(self, sender: str, amount: int) -> None:
        """Deposit ETH into the bank."""
        self.balances[sender] += amount
        self.total_eth += amount
    
    def withdraw(self, sender: str, external_call: Optional[Callable] = None) -> None:
        """Withdraw all ETH - SECURE: uses checks-effects-interactions + reentrancy guard."""
        # Reentrancy guard
        if self._entered:
            self.call_log.append(f"  [BLOCKED] Reentrant call from {sender[:10]}")
            return
        self._entered = True
        
        # CHECKS
        balance = self.balances[sender]
        if balance <= 0:
            self._entered = False
            return
        
        # EFFECTS (state update BEFORE external call)
        self.balances[sender] = 0
        self.total_eth -= balance
        
        self.call_log.append(f"withdraw({sender[:10]}...): sending {balance} ETH")
        
        # INTERACTIONS (external call AFTER state update)
        if external_call:
            external_call(balance)
        
        self._entered = False


# Test the secure version
print("=" * 60)
print("SECURE BANK (Checks-Effects-Interactions)")
print("=" * 60)

secure_bank = SecureBank()
secure_bank.deposit("0xhonest1", 10)
secure_bank.deposit("0xhonest2", 10)
secure_bank.deposit("0xhonest3", 10)
secure_bank.deposit(attacker_addr, 1)

print(f"Bank total: {secure_bank.total_eth} ETH")

# Try the same attack
attacker2 = AttackerContract(secure_bank, attacker_addr)
attacker2.attack()

print(f"\n--- After Attack Attempt ---")
print(f"Attacker stolen: {attacker2.stolen} ETH (only their own deposit!)")
print(f"Bank remaining: {secure_bank.total_eth} ETH (intact!)")

print(f"\nCall trace:")
for log in secure_bank.call_log:
    print(f"  {log}")

### 5.2 Integer Overflow/Underflow

Before Solidity 0.8.0, arithmetic operations could silently overflow or underflow. Since 0.8.0, Solidity includes built-in overflow checking, but understanding this vulnerability remains important.

In [ ]:
class UnsafeUint8:
    """Simulate pre-0.8.0 Solidity uint8 with wrapping arithmetic."""
    
    MAX = 255  # uint8 max
    
    def __init__(self, value: int = 0) -> None:
        """Initialize with wrapping to uint8 range."""
        self.value = value & self.MAX  # Mask to 8 bits
    
    def add(self, other: int) -> 'UnsafeUint8':
        """Add with silent overflow (pre-0.8.0 behavior)."""
        return UnsafeUint8((self.value + other) & self.MAX)
    
    def sub(self, other: int) -> 'UnsafeUint8':
        """Subtract with silent underflow (pre-0.8.0 behavior)."""
        return UnsafeUint8((self.value - other) & self.MAX)


class SafeUint8:
    """Simulate Solidity 0.8.0+ uint8 with overflow checking."""
    
    MAX = 255
    
    def __init__(self, value: int = 0) -> None:
        """Initialize with range checking."""
        if value < 0 or value > self.MAX:
            raise OverflowError(f"Value {value} out of uint8 range [0, {self.MAX}]")
        self.value = value
    
    def add(self, other: int) -> 'SafeUint8':
        """Add with overflow checking."""
        result = self.value + other
        if result > self.MAX:
            raise OverflowError(f"uint8 overflow: {self.value} + {other} = {result} > {self.MAX}")
        return SafeUint8(result)
    
    def sub(self, other: int) -> 'SafeUint8':
        """Subtract with underflow checking."""
        result = self.value - other
        if result < 0:
            raise OverflowError(f"uint8 underflow: {self.value} - {other} = {result} < 0")
        return SafeUint8(result)


print("=" * 60)
print("INTEGER OVERFLOW/UNDERFLOW")
print("=" * 60)

# Demonstrate unsafe overflow
print("\n--- Unsafe uint8 (pre-Solidity 0.8.0) ---")
x = UnsafeUint8(250)
result = x.add(10)
print(f"250 + 10 = {result.value} (expected 260, but wraps!)")

y = UnsafeUint8(0)
result = y.sub(1)
print(f"0 - 1 = {result.value} (expected -1, but wraps to MAX!)")

# Show how this enables attacks
print("\n--- Attack Scenario: Token Balance Underflow ---")
balance = UnsafeUint8(0)
print(f"Attacker balance: {balance.value}")
transferred = balance.sub(1)  # Underflow!
print(f"After transferring 1 token they don't have: {transferred.value}")
print("Attacker now has MAX tokens!")

# Safe version
print("\n--- Safe uint8 (Solidity 0.8.0+) ---")
safe_x = SafeUint8(250)
try:
    safe_x.add(10)
except OverflowError as e:
    print(f"250 + 10 -> {e}")

safe_y = SafeUint8(0)
try:
    safe_y.sub(1)
except OverflowError as e:
    print(f"0 - 1 -> {e}")

### 5.3 Front-Running

Front-running occurs when an attacker observes a pending transaction in the mempool and submits their own transaction with a higher gas price to execute first. This is sometimes called a Miner Extractable Value (MEV) attack.

In [ ]:
@dataclass
class PendingTx:
    """A transaction in the mempool."""
    sender: str
    action: str
    params: Dict[str, Any]
    gas_price: int  # in Gwei
    timestamp: float


class MempoolSimulator:
    """Simulate mempool ordering and front-running."""
    
    def __init__(self) -> None:
        """Initialize empty mempool."""
        self.pending: List[PendingTx] = []
        self.executed: List[PendingTx] = []
    
    def submit_tx(self, tx: PendingTx) -> None:
        """Submit a transaction to the mempool."""
        self.pending.append(tx)
    
    def mine_block(self) -> List[PendingTx]:
        """Mine a block: order by gas price (highest first)."""
        ordered = sorted(self.pending, key=lambda t: t.gas_price, reverse=True)
        self.executed.extend(ordered)
        self.pending = []
        return ordered


# Demonstrate front-running
print("=" * 60)
print("FRONT-RUNNING ATTACK SIMULATION")
print("=" * 60)

mempool = MempoolSimulator()
t = time.time()

# Scenario: DEX trade
# 1. Victim submits a large buy order
victim_tx = PendingTx(
    sender="victim",
    action="buy_token",
    params={"token": "UNI", "amount": 10000, "max_price": 25.0},
    gas_price=50,
    timestamp=t
)
mempool.submit_tx(victim_tx)
print(f"1. Victim submits: Buy 10,000 UNI at max $25, gas: {victim_tx.gas_price} Gwei")

# 2. Attacker sees victim's tx and front-runs with higher gas
frontrun_tx = PendingTx(
    sender="attacker",
    action="buy_token",
    params={"token": "UNI", "amount": 5000, "max_price": 30.0},
    gas_price=200,  # Much higher gas!
    timestamp=t + 0.1
)
mempool.submit_tx(frontrun_tx)
print(f"2. Attacker front-runs: Buy 5,000 UNI at max $30, gas: {frontrun_tx.gas_price} Gwei")

# 3. Attacker also submits a back-run sell
backrun_tx = PendingTx(
    sender="attacker",
    action="sell_token",
    params={"token": "UNI", "amount": 5000, "min_price": 24.0},
    gas_price=30,  # Lower gas, executes after victim
    timestamp=t + 0.2
)
mempool.submit_tx(backrun_tx)
print(f"3. Attacker back-runs: Sell 5,000 UNI at min $24, gas: {backrun_tx.gas_price} Gwei")

# Mine the block
block = mempool.mine_block()

print(f"\n--- Block Execution Order (by gas price) ---")
prices = [20.0, 24.5, 24.5]  # Simulated price impact
for i, tx in enumerate(block):
    print(f"  {i+1}. [{tx.sender:>8}] {tx.action:>10} {tx.params['amount']:>6} UNI | "
          f"gas: {tx.gas_price:>3} Gwei | exec price: ${prices[i]:.2f}")

print(f"\n--- Result ---")
print(f"Attacker bought at: $20.00, sold at: $24.50")
profit = (24.50 - 20.00) * 5000
print(f"Attacker profit: ${profit:,.0f} (sandwich attack)")
print(f"Victim paid $24.50 instead of ~$20.00 (price impact from front-run)")

### 5.4 Vulnerability Summary

| Vulnerability | Impact | Mitigation |
|:---|:---|:---|
| Reentrancy | Drain funds | Checks-Effects-Interactions, ReentrancyGuard |
| Integer Overflow | Create tokens from nothing | Solidity 0.8.0+ (built-in checks) |
| Front-Running | MEV extraction | Commit-reveal, private mempools, batch auctions |
| Access Control | Unauthorized actions | Ownable, Role-based access (OpenZeppelin) |
| Uninitialized Storage | Overwrite critical data | Always initialize, use constructors |

---
## 6. Gas Optimization

Gas optimization is crucial because every operation costs real money. Understanding gas costs helps write efficient contracts.

### EVM Operation Costs
| Operation | Gas Cost | Category |
|:---|:---|:---|
| ADD, SUB | 3 | Arithmetic |
| MUL, DIV | 5 | Arithmetic |
| SLOAD | 2,100 | Storage read |
| SSTORE (new) | 20,000 | Storage write |
| SSTORE (update) | 5,000 | Storage write |
| MLOAD, MSTORE | 3 | Memory |
| CALL | 2,600+ | External call |

In [ ]:
class GasProfiler:
    """Profile gas costs for different coding patterns."""
    
    # Gas costs per opcode
    COSTS = {
        "ADD": 3, "SUB": 3, "MUL": 5, "DIV": 5, "MOD": 5,
        "LT": 3, "GT": 3, "EQ": 3, "AND": 3, "OR": 3,
        "SLOAD": 2_100, "SSTORE_NEW": 20_000, "SSTORE_UPDATE": 5_000,
        "MLOAD": 3, "MSTORE": 3,
        "PUSH": 3, "POP": 2, "DUP": 3, "SWAP": 3,
        "CALL": 2_600, "STATICCALL": 2_600,
        "KECCAK256": 30, "LOG": 375,
    }
    
    def __init__(self, label: str) -> None:
        """Initialize gas profiler with a label."""
        self.label = label
        self.ops: List[Tuple[str, int]] = []
        self.total = 0
    
    def op(self, name: str, count: int = 1) -> 'GasProfiler':
        """Record an operation."""
        cost = self.COSTS.get(name, 0) * count
        self.ops.append((f"{name} x{count}" if count > 1 else name, cost))
        self.total += cost
        return self
    
    def summary(self) -> str:
        """Return gas summary string."""
        lines = [f"\n{self.label}:"]
        for name, cost in self.ops:
            lines.append(f"  {name:<20} {cost:>8,} gas")
        lines.append(f"  {'TOTAL':<20} {self.total:>8,} gas")
        return "\n".join(lines)


print("=" * 60)
print("GAS OPTIMIZATION COMPARISONS")
print("=" * 60)

# Comparison 1: Storage vs Memory
print("\n--- Pattern 1: Storage Access Optimization ---")

# Bad: Reading storage in a loop
bad = GasProfiler("BAD: Read storage each iteration (5 iterations)")
bad.op("SLOAD", 5)  # Read storage variable 5 times
bad.op("ADD", 5)
bad.op("SSTORE_UPDATE", 5)  # Write storage 5 times

# Good: Cache in memory, write once
good = GasProfiler("GOOD: Cache in memory, write once")
good.op("SLOAD", 1)  # Read once
good.op("MSTORE", 1)  # Cache in memory
good.op("MLOAD", 5)  # Read from memory in loop
good.op("ADD", 5)
good.op("MSTORE", 5)  # Update memory
good.op("SSTORE_UPDATE", 1)  # Write once

print(bad.summary())
print(good.summary())
savings1 = bad.total - good.total
print(f"\n  Savings: {savings1:,} gas ({savings1/bad.total*100:.1f}%)")

In [ ]:
# Comparison 2: Variable Packing
print("=" * 60)
print("STORAGE VARIABLE PACKING")
print("=" * 60)

# In Solidity, each storage slot is 32 bytes.
# Variables smaller than 32 bytes can be packed into a single slot.

@dataclass
class UnpackedStruct:
    """Unpacked: each field uses its own 32-byte slot."""
    # Slot 0: uint256 (32 bytes)
    amount: int = 0         # 32 bytes -> slot 0
    # Slot 1: uint8 (padded to 32 bytes)
    status: int = 0         # 1 byte padded to 32 -> slot 1
    # Slot 2: address (padded to 32 bytes)
    owner: str = ""         # 20 bytes padded to 32 -> slot 2
    # Slot 3: uint8 (padded to 32 bytes)
    category: int = 0       # 1 byte padded to 32 -> slot 3

@dataclass
class PackedStruct:
    """Packed: smaller variables share slots."""
    # Slot 0: uint256 (32 bytes) - can't pack with others
    amount: int = 0         # 32 bytes -> slot 0
    # Slot 1: address(20) + uint8(1) + uint8(1) = 22 bytes (fits in one slot!)
    owner: str = ""         # 20 bytes |
    status: int = 0         # 1 byte   | -> slot 1
    category: int = 0       # 1 byte   |

unpacked_slots = 4  # Each field in its own slot
packed_slots = 2    # Packed into 2 slots

unpacked_gas = unpacked_slots * 20_000  # SSTORE_NEW for each slot
packed_gas = packed_slots * 20_000

print(f"\nUnpacked struct: {unpacked_slots} storage slots")
print(f"  Layout: [amount:32] [status:1+pad:31] [owner:20+pad:12] [category:1+pad:31]")
print(f"  Write cost: {unpacked_gas:,} gas")

print(f"\nPacked struct: {packed_slots} storage slots")
print(f"  Layout: [amount:32] [owner:20|status:1|category:1|pad:10]")
print(f"  Write cost: {packed_gas:,} gas")

savings = unpacked_gas - packed_gas
print(f"\nSavings: {savings:,} gas ({savings/unpacked_gas*100:.1f}%)")
print(f"At 50 Gwei and ETH=$2000: saves ${savings * 50e-9 * 2000:.2f} per write")

In [ ]:
# Comparison 3: Short-circuit evaluation and other patterns
print("=" * 60)
print("ADDITIONAL GAS OPTIMIZATION PATTERNS")
print("=" * 60)

patterns = [
    ("Use != 0 instead of > 0 for uint", 3, 3, 
     "Both cost 3 gas, but != 0 is convention"),
    ("Use ++i instead of i++", 3, 5,
     "Post-increment creates temp variable"),
    ("Use immutable for deploy-time constants", 2_100, 3,
     "SLOAD vs PUSH (constant baked into bytecode)"),
    ("Use custom errors instead of strings", 375 + 30, 30,
     "Error strings stored in runtime code, custom errors are 4 bytes"),
    ("Use mapping instead of array for lookups", 2_100 + 30 + 3, 2_100,
     "Arrays require index bounds check + offset calculation"),
    ("Batch operations (1 SSTORE vs 5)", 5 * 5_000, 5_000,
     "Accumulate in memory, write once to storage"),
]

print(f"\n{'Pattern':<45} {'Before':>8} {'After':>8} {'Savings':>8}")
print("-" * 75)
for name, before, after, note in patterns:
    savings = before - after
    print(f"{name:<45} {before:>7,}  {after:>7,}  {savings:>7,}")
    print(f"  {'Note:':<7} {note}")

---
## 7. Contract Testing Framework

Testing is essential for smart contracts since they're immutable once deployed. Let's build a simple test framework and write tests for our ERC-20 implementation.

In [ ]:
class ContractTestFramework:
    """Simple test framework for smart contract simulators."""
    
    def __init__(self, name: str) -> None:
        """Initialize test suite."""
        self.name = name
        self.passed = 0
        self.failed = 0
        self.results: List[Tuple[str, bool, str]] = []
    
    def assert_equal(self, actual: Any, expected: Any, test_name: str) -> None:
        """Assert two values are equal."""
        if actual == expected:
            self.passed += 1
            self.results.append((test_name, True, ""))
        else:
            self.failed += 1
            self.results.append((test_name, False, f"expected {expected}, got {actual}"))
    
    def assert_true(self, condition: bool, test_name: str) -> None:
        """Assert a condition is true."""
        self.assert_equal(condition, True, test_name)
    
    def assert_reverts(self, func: Callable, test_name: str, 
                       error_msg: Optional[str] = None) -> None:
        """Assert that a function call reverts."""
        try:
            func()
            self.failed += 1
            self.results.append((test_name, False, "did not revert"))
        except Exception as e:
            if error_msg and error_msg not in str(e):
                self.failed += 1
                self.results.append((test_name, False, f"wrong error: {e}"))
            else:
                self.passed += 1
                self.results.append((test_name, True, ""))
    
    def report(self) -> None:
        """Print test results."""
        print(f"\n{'=' * 60}")
        print(f"TEST SUITE: {self.name}")
        print(f"{'=' * 60}")
        for name, passed, msg in self.results:
            status = "PASS" if passed else "FAIL"
            icon = "+" if passed else "x"
            line = f"  [{icon}] {status}: {name}"
            if msg:
                line += f" ({msg})"
            print(line)
        print(f"\n  Results: {self.passed} passed, {self.failed} failed, "
              f"{self.passed + self.failed} total")
        if self.failed == 0:
            print("  All tests passed!")

print("Test framework defined.")

In [ ]:
# Run comprehensive tests on our ERC-20 implementation
t = ContractTestFramework("ERC-20 Token")

# Setup
DECIMALS = 18
SUPPLY = 1_000_000 * 10**DECIMALS
alice = "0x" + "a1" * 20
bob = "0x" + "b2" * 20
charlie = "0x" + "c3" * 20
zero = "0x" + "00" * 20

# Test deployment
token = ERC20(alice, "TestToken", "TT", DECIMALS, SUPPLY)
t.assert_equal(token.name(), "TestToken", "name returns correct value")
t.assert_equal(token.symbol(), "TT", "symbol returns correct value")
t.assert_equal(token.decimals(), 18, "decimals returns 18")
t.assert_equal(token.total_supply(), SUPPLY, "total supply matches initial")
t.assert_equal(token.balance_of(alice), SUPPLY, "deployer gets initial supply")
t.assert_equal(token.balance_of(bob), 0, "other addresses start at 0")

# Test transfer
amount = 1000 * 10**DECIMALS
token.call_as(alice).transfer(bob, amount)
t.assert_equal(token.balance_of(alice), SUPPLY - amount, "sender balance decreases")
t.assert_equal(token.balance_of(bob), amount, "receiver balance increases")
t.assert_equal(token.total_supply(), SUPPLY, "total supply unchanged after transfer")

# Test transfer failures
t.assert_reverts(
    lambda: token.call_as(bob).transfer(alice, amount * 1000),
    "transfer reverts on insufficient balance",
    "exceeds balance"
)
t.assert_reverts(
    lambda: token.call_as(alice).transfer(zero, amount),
    "transfer reverts on zero address",
    "zero address"
)

# Test approve and transferFrom
allowance = 500 * 10**DECIMALS
token.call_as(alice).approve(charlie, allowance)
t.assert_equal(token.allowance(alice, charlie), allowance, "allowance set correctly")

spend = 200 * 10**DECIMALS
alice_before = token.balance_of(alice)
token.call_as(charlie).transfer_from(alice, bob, spend)
t.assert_equal(token.balance_of(alice), alice_before - spend, "transferFrom decreases from")
t.assert_equal(token.allowance(alice, charlie), allowance - spend, "allowance decreases")

t.assert_reverts(
    lambda: token.call_as(charlie).transfer_from(alice, bob, allowance),
    "transferFrom reverts on insufficient allowance",
    "insufficient allowance"
)

# Test mint and burn
mint_amount = 100 * 10**DECIMALS
token._mint(bob, mint_amount)
t.assert_equal(token.total_supply(), SUPPLY + mint_amount, "mint increases total supply")

token._burn(bob, mint_amount)
t.assert_equal(token.total_supply(), SUPPLY, "burn decreases total supply")

t.assert_reverts(
    lambda: token._burn(bob, SUPPLY * 10),
    "burn reverts on insufficient balance",
    "exceeds balance"
)

t.report()

---
## 8. Putting It All Together: A Complete Token with Access Control

Let's combine our patterns into a production-quality token that includes ownership, pausing, minting caps, and event tracking.

In [ ]:
class ManagedToken(ERC20):
    """ERC-20 token with Ownable + Pausable + minting cap."""
    
    def __init__(self, deployer: str, name: str, symbol: str,
                 max_supply: int, initial_supply: int) -> None:
        """Deploy managed token with max supply cap."""
        super().__init__(deployer, name, symbol, 18, initial_supply)
        self._owner = deployer
        self._paused = False
        self._max_supply = max_supply
        self._minters: Dict[str, bool] = {deployer: True}
    
    def transfer(self, to: str, amount: int) -> bool:
        """Transfer with pause check."""
        self._require(not self._paused, "Token is paused")
        return super().transfer(to, amount)
    
    def mint(self, to: str, amount: int) -> None:
        """Mint new tokens (minters only, respects max supply)."""
        self._require(self._minters.get(self.msg_sender, False), "Not a minter")
        self._require(not self._paused, "Token is paused")
        self._require(self._total_supply + amount <= self._max_supply, 
                     f"Would exceed max supply of {self._max_supply}")
        self._mint(to, amount)
    
    def add_minter(self, minter: str) -> None:
        """Add a new minter address (owner only)."""
        self._require(self.msg_sender == self._owner, "Only owner")
        self._minters[minter] = True
        self._emit_event("MinterAdded", minter=minter[:12])
    
    def pause(self) -> None:
        """Pause the token (owner only)."""
        self._require(self.msg_sender == self._owner, "Only owner")
        self._paused = True
        self._emit_event("Paused")
    
    def unpause(self) -> None:
        """Unpause the token (owner only)."""
        self._require(self.msg_sender == self._owner, "Only owner")
        self._paused = False
        self._emit_event("Unpaused")


# Demo
DECIMALS = 18
MAX_SUPPLY = 10_000_000 * 10**DECIMALS
INITIAL = 1_000_000 * 10**DECIMALS

admin = "0x" + "a1" * 20
user1 = "0x" + "b2" * 20
user2 = "0x" + "c3" * 20

managed = ManagedToken(admin, "ManagedCoin", "MGD", MAX_SUPPLY, INITIAL)

print("=" * 60)
print("MANAGED TOKEN (Combined Patterns)")
print("=" * 60)
print(f"Name: {managed.name()} ({managed.symbol()})")
print(f"Initial supply: {managed._format_amount(managed.total_supply())}")
print(f"Max supply: {managed._format_amount(MAX_SUPPLY)}")

# Normal operations
managed.call_as(admin).transfer(user1, 100_000 * 10**DECIMALS)
print(f"\nTransferred 100,000 to user1")

# Minting
managed.call_as(admin).mint(user2, 500_000 * 10**DECIMALS)
print(f"Minted 500,000 to user2")
print(f"Total supply: {managed._format_amount(managed.total_supply())}")

# Pause and try transfer
managed.call_as(admin).pause()
print(f"\nToken PAUSED")
try:
    managed.call_as(user1).transfer(user2, 1000 * 10**DECIMALS)
except Exception as e:
    print(f"Transfer blocked: {e}")

# Unpause
managed.call_as(admin).unpause()
managed.call_as(user1).transfer(user2, 1000 * 10**DECIMALS)
print(f"\nToken UNPAUSED - transfer successful")

# Non-minter cannot mint
try:
    managed.call_as(user1).mint(user1, 1000 * 10**DECIMALS)
except Exception as e:
    print(f"Non-minter mint: {e}")

---
## Exercises

### Exercise 1: ERC-20 Extension - Burnable Token

Create a `BurnableToken` that extends our `ERC20` class. Add a public `burn()` function that allows any holder to burn their own tokens, and a `burn_from()` function that uses allowances.

**Hints:**
- Use the existing `_burn()` internal function
- `burn_from()` should check and update allowances like `transfer_from()`
- Remember to verify the caller has sufficient balance

In [ ]:
class BurnableToken(ERC20):
    """ERC-20 token with public burn functionality."""
    
    def burn(self, amount: int) -> None:
        """Burn tokens from msg.sender's balance."""
        # YOUR CODE HERE
        pass
    
    def burn_from(self, account: str, amount: int) -> None:
        """Burn tokens from another account using allowance."""
        # YOUR CODE HERE
        pass

# Test your implementation
# token = BurnableToken("0x" + "a1" * 20, "BurnCoin", "BURN", 18, 1000000 * 10**18)
# token.call_as("0x" + "a1" * 20).burn(100 * 10**18)
# print(f"Supply after burn: {token._format_amount(token.total_supply())}")

### Exercise 2: Access Control with Roles

Implement a `RoleBasedAccess` contract that supports multiple roles (ADMIN, MINTER, PAUSER) instead of just a single owner. Each role can have multiple members.

**Hints:**
- Use a nested dict: `_roles[role_name][address] = bool`
- Only ADMIN should be able to grant/revoke roles
- The deployer starts as the only ADMIN

In [ ]:
class RoleBasedAccess(ContractBase):
    """Contract with role-based access control."""
    
    ADMIN_ROLE = "ADMIN"
    MINTER_ROLE = "MINTER"
    PAUSER_ROLE = "PAUSER"
    
    def __init__(self, deployer: str) -> None:
        """Deploy with deployer as initial ADMIN."""
        super().__init__(deployer)
        # YOUR CODE HERE
        pass
    
    def has_role(self, role: str, account: str) -> bool:
        """Check if an account has a specific role."""
        # YOUR CODE HERE
        pass
    
    def grant_role(self, role: str, account: str) -> None:
        """Grant a role to an account (ADMIN only)."""
        # YOUR CODE HERE
        pass
    
    def revoke_role(self, role: str, account: str) -> None:
        """Revoke a role from an account (ADMIN only)."""
        # YOUR CODE HERE
        pass

### Exercise 3: Gas Cost Calculator

Build a function that estimates the gas cost of a typical ERC-20 transfer, breaking it down by operation type. Account for: signature verification, balance reads, balance writes, allowance check (if transferFrom), and event emission.

**Hints:**
- A basic transfer needs: 2 SLOAD (balances), 2 SSTORE (balances), LOG
- transferFrom adds: 1 SLOAD (allowance), 1 SSTORE (allowance)
- Base transaction cost is 21,000 gas

In [ ]:
def estimate_transfer_gas(is_transfer_from: bool = False,
                          is_first_transfer: bool = False) -> Dict[str, int]:
    """Estimate gas cost for an ERC-20 transfer operation.
    
    Args:
        is_transfer_from: Whether this is a transferFrom (uses allowance)
        is_first_transfer: Whether receiver has never received tokens before
    
    Returns:
        Dictionary with gas breakdown by category
    """
    # YOUR CODE HERE
    pass

# Test:
# result = estimate_transfer_gas(is_transfer_from=False)
# for category, cost in result.items():
#     print(f"{category}: {cost:,} gas")

### Exercise 4: Detect Vulnerable Patterns

Write a `VulnerabilityScanner` class that analyzes a Python contract class and flags potential vulnerabilities based on code patterns.

**Hints:**
- Check if external calls (callbacks) happen before state updates
- Check if there's a reentrancy guard
- Check if arithmetic operations have overflow protection
- Use Python's `inspect` module to read source code

In [ ]:
import inspect

class VulnerabilityScanner:
    """Scan Python contract simulators for vulnerability patterns."""
    
    def scan(self, contract_class: type) -> List[Dict[str, str]]:
        """Scan a contract class for vulnerability patterns.
        
        Returns list of findings with 'severity', 'type', and 'description'.
        """
        # YOUR CODE HERE
        pass

# Test:
# scanner = VulnerabilityScanner()
# findings = scanner.scan(VulnerableBank)
# for f in findings:
#     print(f"[{f['severity']}] {f['type']}: {f['description']}")

---
## Summary

### What You Learned
- [x] Solidity data types (uint256, address, mapping) and their constraints
- [x] EVM storage model with gas costs for reads and writes
- [x] Contract design patterns: Ownable, Pausable, ReentrancyGuard
- [x] Complete ERC-20 token implementation with transfer, approve, transferFrom
- [x] ERC-721 NFT standard with minting and ownership tracking
- [x] Critical vulnerabilities: reentrancy, integer overflow, front-running
- [x] Gas optimization techniques: storage packing, caching, batching
- [x] Contract testing methodology and test framework design

### Key Takeaways
1. **Security first**: Always follow Checks-Effects-Interactions and use ReentrancyGuard
2. **Gas matters**: Storage operations dominate gas costs - optimize storage access
3. **Standards enable composability**: ERC-20/721 allow tokens to work across the ecosystem
4. **Test everything**: Contracts are immutable - bugs can't be patched after deployment

### Next Steps
- [Notebook 05: DeFi Protocols](05-defi-protocols.ipynb) - Build on token standards to explore AMMs, lending, and yield farming
- [Section 03: Ethereum & Smart Contracts](../sections/03-ethereum-smart-contracts.md) - Deeper reading on the Ethereum ecosystem